# 04 - Transcriptomic Analysis with CGCS

**Simulated RNA-seq / Gene Expression Data**  
**Allen Lab Trisomy 21 Collaboration**

In [ ]:
# ================================================
# SETUP
# ================================================
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

import grok
from grok.trisomy_metrics import trisomy_cgcs_score
from grok.visualization import plot_cgcs_vs_noise

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Transcriptomic Notebook Ready")
print(f"Version: {grok.__version__}")

## 1. Simulate RNA-seq Style Data (Normal vs Trisomy 21)

In [ ]:
np.random.seed(42)

# Simulate 500 genes
n_genes = 500

# Normal expression (log2 scale)
normal_expr = np.random.normal(8.0, 2.0, n_genes)

# Trisomy 21: ~1.5x dosage on ~300 genes (chr21-like), normal on others
trisomy_expr = normal_expr.copy()
chr21_like_genes = np.random.choice(n_genes, 80, replace=False)  # ~16% like real chr21
trisomy_expr[chr21_like_genes] *= 1.5

# Add some global dysregulation noise
trisomy_expr += np.random.normal(0, 0.3, n_genes)

print(f"Simulated {n_genes} genes")
print(f"Mean expression Normal : {normal_expr.mean():.3f}")
print(f"Mean expression Trisomy: {trisomy_expr.mean():.3f}")

## 2. CGCS on Transcriptomic Data

In [ ]:
def transcriptomic_cgcs(expr_normal, expr_trisomy, overexpression_weight=0.6):
    """Simple CGCS proxy from expression data"""
    fold_change = expr_trisomy / (expr_normal + 1e-6)
    mean_fc = np.mean(fold_change)
    imbalance = np.abs(mean_fc - 1.0)
    dysregulation = np.std(fold_change)
    
    return trisomy_cgcs_score(
        dosage_ratio=mean_fc,
        overexpression_imbalance=imbalance * overexpression_weight,
        global_dysregulation=dysregulation,
        return_components=True
    )

result = transcriptomic_cgcs(normal_expr, trisomy_expr)

print("Transcriptomic CGCS Results:")
for k, v in result.items():
    print(f"  {k:25} = {v:.4f}")

## 3. Visualization

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(normal_expr, color='blue', alpha=0.6, label='Normal', kde=True)
sns.histplot(trisomy_expr, color='red', alpha=0.6, label='Trisomy 21', kde=True)
plt.title("Simulated Gene Expression Distribution")
plt.xlabel("Log2 Expression")
plt.ylabel("Gene Count")
plt.legend()
plt.show()

## 4. CGCS vs Overexpression Level

In [ ]:
overexpression_levels = np.linspace(1.0, 1.8, 15)
cgcs_scores = []

for level in overexpression_levels:
    temp_expr = normal_expr.copy()
    temp_expr[chr21_like_genes] *= level
    score = transcriptomic_cgcs(normal_expr, temp_expr)['cgcs']
    cgcs_scores.append(score)

plot_cgcs_vs_noise(
    noise_levels=overexpression_levels,
    cgcs_values=cgcs_scores,
    title="CGCS vs Overexpression Level (Transcriptomic View)"
)

---
**Next Steps**

- Integrate real Allen Lab RNA-seq data
- Build per-gene CGCS
- Model specific drug interventions (e.g. DYRK1A inhibition)